# RePaint on any image — paper-style qualitative results
*Lugmayr et al., "RePaint: Inpainting using Denoising Diffusion Probabilistic Models", CVPR 2022 · [arXiv:2201.09865](https://arxiv.org/abs/2201.09865)*

This notebook reproduces the paper's **qualitative figures** on **your own images**:

| Figure | What it shows | Paper reference |
|---|---|---|
| **A** | Input → denoising 0 % / 60 % / 75 % → 5 diverse samples, one row per image | Fig. 1 (teaser) |
| **B** | One image under all six mask types, several samples each | Figs. 5–7 and supplementary qualitative grids |
| **C** | Without resampling (r = 1) vs. RePaint (r = 10), same seed | Fig. 3 / Sec. 4.2 (harmonisation) |
| **D** | Animated reverse process, including the forward "jumps" | Fig. 2 / the resampling schedule |
| **E** | Diversity and fidelity: pairwise LPIPS between samples, LPIPS to the original | Sec. 5 (diversity discussion) |

**Which prior is used.** RePaint is model-agnostic. The paper uses a different pretrained DDPM per domain, and so does this notebook:

| Domain | Checkpoint (same as the paper) | Source |
|---|---|---|
| `face` | `celeba256_250000.pt` (CelebA-HQ) | authors' Google Drive (`download.sh`) → fallback `google/ddpm-ema-celebahq-256` |
| `places` | `places256_300000.pt` (Places2) | authors' Google Drive → fallback `imagenet` |
| `imagenet` (any other image) | OpenAI `256x256_diffusion.pt` + `256x256_classifier.pt`, class-conditional with classifier guidance (scale 1.0) | `openaipublic.blob.core.windows.net` |
| `imagenet_uncond` | OpenAI `256x256_diffusion_uncond.pt` | `openaipublic.blob.core.windows.net` |

`DOMAIN = "auto"` sends an image to `face` when a face is detected and to `imagenet` otherwise.

The sampler is the same verified implementation as the benchmark notebook: T = 250, j = 10, r = 10, learned variance, and the official jump schedule (2410 network evaluations per sample).

**Kaggle:** GPU T4 ×2, Internet On. Add your own images as a Kaggle dataset, or leave `IMAGE_DIR = None` to use a built-in demo set (a face, a cat, coffee, a rocket).

## 0 · Configuration — **edit this cell**

In [ ]:
# ============================== W&B =========================================
WANDB_API_KEY  = ""                                  # <-- paste your W&B key (or Kaggle Secret WANDB_API_KEY)
WANDB_PROJECT  = "repaint-qualitative-figures"       # <-- change project name
WANDB_ENTITY   = None
WANDB_RUN_NAME = "fig1-style-any-image"              # <-- change run name

# ============================== Inputs ======================================
IMAGE_DIR = None        # folder with your images (e.g. "/kaggle/input/my-photos"); None = built-in demo images
MASK_DIR  = None        # optional folder of masks with the SAME file stem as the image (e.g. cat.png <-> cat.png)
MASK_WHITE_MEANS = "known"   # official RePaint convention: white (255) = keep, black (0) = inpaint
MAX_IMAGES = 4
# Mask for images without a mask file: "auto" | "face_lower" | "box" | "wide" | "narrow" | "sr2x" | "altlines" | "half" | "expand"
#   "auto" = lower-face mask for faces; otherwise cycles narrow / expand / half / wide, like the rows of paper Fig. 1
DEFAULT_MASK = "auto"
PER_IMAGE_MASK = {}     # e.g. {"cat": "half", "street": "wide"} keyed by file stem

# ============================== Prior =======================================
DOMAIN = "auto"         # "auto" | "face" | "places" | "imagenet" | "imagenet_uncond"
PER_IMAGE_DOMAIN = {}   # e.g. {"street": "places"}
CLASS_LABEL = "auto"    # imagenet only: "auto" (classify the visible context), "random" (official test.py protocol) or an int 0-999
CLASSIFIER_SCALE = 1.0  # paper / official config value
FACE_ALIGN = True       # crop detected faces to CelebA-HQ framing before inpainting

# ============================== RePaint =====================================
T_STEPS, JUMP_LENGTH, JUMP_N_SAMPLE = 250, 10, 10     # paper settings
N_SAMPLES = 5                                         # Fig. 1 shows 5 samples per input
DENOISE_SNAPSHOTS = [0.0, 0.60, 0.75]                 # Fig. 1 columns "Denoising 0% / 60% / 75%"
SEED = 2022

# ============================== Extra figures ===============================
RUN_RESAMPLING_COMPARISON = True    # Figure C: r = 1 vs r = 10 (cheap)
RUN_MASK_GALLERY = True             # Figure B: one image x 6 mask types
GALLERY_IMAGE = 0                   # index of the image used for Figure B
GALLERY_SAMPLES = 2
MAKE_GIFS = True                    # Figure D

# ============================== Compute =====================================
TIME_BUDGET_HOURS = 9.5
MAX_BATCH_PER_GPU = None            # None = auto
USE_FP16 = True
DRY_RUN = False                     # True = tiny step counts for a quick smoke test

REPAINT_REPO = "https://github.com/andreas128/RePaint.git"
GDRIVE_CKPT_IDS = ["1norNWWGYP3EZ_o05DmoW1ryKuKMmhlCX", "1QEl-btGbzQz6IwkXiFGd49uQNTUtTHsk"]  # celeba256 + places256
OPENAI_URL = "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/"

## 1 · Setup

In [ ]:
import subprocess, sys, importlib, os
def _pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=False)
need = [p for p, m in [("lpips", "lpips"), ("wandb", "wandb"), ("diffusers", "diffusers"), ("gdown", "gdown"),
                       ("opencv-python-headless", "cv2"), ("scikit-image", "skimage"), ("imageio", "imageio")]
        if importlib.util.find_spec(m) is None]
if need: _pip(*need)
from pathlib import Path
import shutil, zipfile, time, math, random, json, threading, queue, warnings, types, urllib.request
import numpy as np, pandas as pd, torch, torch.nn.functional as F, cv2, imageio
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
START = time.time()
INPUT_ROOT   = Path(os.environ.get("REPAINT_INPUT_ROOT", "/kaggle/input"))
WORKING_ROOT = Path(os.environ.get("REPAINT_WORKING_ROOT", "/kaggle/working")); WORKING_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORKING_ROOT / "RePaint"
if not (REPO_DIR / "guided_diffusion").exists():
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", REPAINT_REPO, str(REPO_DIR)], capture_output=True, text=True)
    print("git clone:", "ok" if r.returncode == 0 else r.stderr[-300:])
sys.path.insert(0, str(REPO_DIR))
OUT = WORKING_ROOT / "repaint_figures"; FIG = OUT / "figures"; SAMPLES = OUT / "samples"; ASSETS = WORKING_ROOT / "repaint_assets"
for d in (OUT, FIG, SAMPLES, ASSETS, OUT / "gifs", OUT / "cache"): d.mkdir(parents=True, exist_ok=True)

DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())] or [torch.device("cpu")]
MAIN = DEVICES[0]; IS_CUDA = MAIN.type == "cuda"; USE_FP16 = USE_FP16 and IS_CUDA
if IS_CUDA: torch.backends.cudnn.benchmark = True
if DRY_RUN: T_STEPS, JUMP_LENGTH, JUMP_N_SAMPLE, N_SAMPLES, GALLERY_SAMPLES = 20, 5, 2, 3, 1
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
def hours_left(): return TIME_BUDGET_HOURS - (time.time() - START) / 3600
print(f"{len(DEVICES)} device(s):", ", ".join(torch.cuda.get_device_name(d) for d in DEVICES) if IS_CUDA else "cpu")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight", "font.family": "serif", "font.size": 9})

## 2 · Weights & Biases

In [ ]:
import wandb
key = (WANDB_API_KEY or "").strip()
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY") or ""
    except Exception: key = ""
os.environ.setdefault("WANDB_DIR", str(WORKING_ROOT)); os.environ["WANDB_SILENT"] = "true"
WANDB_MODE = "online" if key else "offline"
if key: wandb.login(key=key, relogin=True)
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=WANDB_RUN_NAME, mode=WANDB_MODE,
                 config=dict(T=T_STEPS, j=JUMP_LENGTH, r=JUMP_N_SAMPLE, n_samples=N_SAMPLES, domain=DOMAIN,
                             class_label=str(CLASS_LABEL), classifier_scale=CLASSIFIER_SCALE, dry_run=DRY_RUN))
print("W&B:", run.url if WANDB_MODE == "online" else f"offline ({run.dir})")
def wlog(d):
    if wandb.run is not None: wandb.log(d)
def save_fig(fig, key, pdf=True):
    p = FIG / f"{key.replace('/', '__')}.png"; fig.savefig(p)
    if pdf: fig.savefig(p.with_suffix(".pdf"))          # vector copy for reports / papers
    wlog({key: wandb.Image(str(p))}); plt.show(); plt.close(fig)

## 3 · Inputs: images, face detection, domain routing and masks
* **Preprocessing.** Generic images use the official `center_crop_arr` (shorter side to 256, then a centre crop). With `FACE_ALIGN`, a detected face is re-framed like CelebA-HQ: a square of about 2.1× the face width, centred slightly above the nose.
* **Masks.** These are the paper's six distributions (LaMa Wide/Narrow, Super-Resolve 2×, Alternating Lines, Half, Expand), plus `face_lower` (a mouth-and-chin region like the Fig. 1 face row), `box`, or your own mask files.

In [ ]:
import skimage.data as skdata
OPENCV_CASCADE = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml" if hasattr(cv2, "data") else None

def detect_face(rgb):
    """Largest frontal face (x, y, w, h) or None. OpenCV Haar cascade; falls back to scikit-image's LBP cascade."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    if OPENCV_CASCADE is not None and OPENCV_CASCADE.exists():
        faces = cv2.CascadeClassifier(str(OPENCV_CASCADE)).detectMultiScale(gray, 1.1, 5, minSize=(40, 40))
        faces = [tuple(map(int, f)) for f in faces]
    else:
        from skimage.feature import Cascade
        det = Cascade(skdata.lbp_frontal_face_cascade_filename())
        faces = [(d["c"], d["r"], d["width"], d["height"]) for d in
                 det.detect_multi_scale(img=rgb, scale_factor=1.2, step_ratio=1, min_size=(40, 40), max_size=(2000, 2000))]
    return max(faces, key=lambda f: f[2] * f[3]) if faces else None

def center_crop_arr(pil, size=256):            # official RePaint / guided-diffusion preprocessing
    while min(*pil.size) >= 2 * size:
        pil = pil.resize(tuple(x // 2 for x in pil.size), resample=Image.BOX)
    s = size / min(*pil.size); pil = pil.resize(tuple(round(x * s) for x in pil.size), resample=Image.BICUBIC)
    a = np.array(pil); cy, cx = (a.shape[0] - size) // 2, (a.shape[1] - size) // 2
    return a[cy:cy + size, cx:cx + size]

def face_crop(rgb, face, size=256):
    x, y, w, h = face; side = 2.1 * w; cx, cy = x + w / 2, y + 0.45 * h
    pad = int(side)
    big = cv2.copyMakeBorder(rgb, pad, pad, pad, pad, cv2.BORDER_REFLECT_101)
    x0, y0 = int(round(cx - side / 2)) + pad, int(round(cy - side / 2)) + pad
    crop = big[y0:y0 + int(side), x0:x0 + int(side)]
    return np.array(Image.fromarray(crop).resize((size, size), Image.LANCZOS))

# ---- load images ----
raw = []
if IMAGE_DIR:
    for p in sorted(Path(IMAGE_DIR).rglob("*")):
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}:
            raw.append((p.stem, np.array(Image.open(p).convert("RGB"))))
else:
    raw = [("astronaut", skdata.astronaut()), ("cat", skdata.chelsea()), ("coffee", skdata.coffee()), ("rocket", skdata.rocket())]
raw = raw[:MAX_IMAGES]
assert raw, "No images found"

ITEMS = []   # one dict per input image
for i, (name, rgb) in enumerate(raw):
    face = detect_face(rgb)
    dom = PER_IMAGE_DOMAIN.get(name, DOMAIN)
    if dom == "auto": dom = "face" if face is not None else "imagenet"
    img = face_crop(rgb, face) if (face is not None and FACE_ALIGN and dom == "face") else center_crop_arr(Image.fromarray(rgb))
    ITEMS.append(dict(name=name, img=img, domain=dom, face=face is not None))
print(pd.DataFrame([{k: v for k, v in it.items() if k != "img"} for it in ITEMS]))

In [ ]:
# ---- masks (1 = known, 0 = inpaint) ----
def _irregular(rng, H, W, max_angle, max_len, max_width, min_times, max_times):
    m = np.zeros((H, W), np.float32)
    for i in range(rng.randint(min_times, max_times + 1)):
        sx, sy = rng.randint(W), rng.randint(H)
        for _ in range(1 + rng.randint(5)):
            a = 0.01 + rng.randint(max_angle); a = 2 * 3.1415926 - a if i % 2 == 0 else a
            L, bw = 10 + rng.randint(max_len), 5 + rng.randint(max_width)
            ex, ey = int(np.clip(sx + L * np.sin(a), 0, W)), int(np.clip(sy + L * np.cos(a), 0, H))
            cv2.line(m, (sx, sy), (ex, ey), 1.0, bw); sx, sy = ex, ey
    return m
def _boxes(rng, H, W, margin, bmin, bmax, mn, mx):
    m = np.zeros((H, W), np.float32); bmax = min(bmax, H - 2 * margin, W - 2 * margin)
    for _ in range(rng.randint(mn, mx + 1)):
        bw, bh = rng.randint(bmin, bmax), rng.randint(bmin, bmax)
        x0, y0 = rng.randint(margin, W - margin - bw + 1), rng.randint(margin, H - margin - bh + 1); m[y0:y0 + bh, x0:x0 + bw] = 1
    return m

def make_mask(kind, seed, H=256, W=256):
    rng = np.random.RandomState(seed); hole = np.zeros((H, W), np.float32)
    if kind in ("wide", "narrow"):
        for _ in range(100):
            if kind == "wide":
                hole = _irregular(rng, H, W, 4, 200, 100, 1, 5) if rng.rand() < 1 / 1.3 else _boxes(rng, H, W, 10, 30, 150, 1, 3)
            else:
                hole = _irregular(rng, H, W, 4, 40, 10, 4, 50)
            if 0 < hole.mean() <= 0.5: break
    elif kind == "sr2x": hole[:] = 1; hole[::2, ::2] = 0
    elif kind == "altlines": hole[1::2, :] = 1
    elif kind == "half": hole[:, W // 2:] = 1
    elif kind == "expand": hole[:] = 1; c = (H - 64) // 2; hole[c:c + 64, c:c + 64] = 0
    elif kind == "box": c = int(H * 0.3); hole[c:H - c, c:W - c] = 1
    elif kind == "face_lower":                    # mouth + chin region, like the face row of Fig. 1
        cv2.ellipse(hole, (W // 2, int(H * 0.72)), (int(W * 0.25), int(H * 0.19)), 0, 0, 360, 1.0, -1)
    else: raise ValueError(kind)
    return 1.0 - hole

MASK_TYPES = ["wide", "narrow", "sr2x", "altlines", "half", "expand"]
MASK_LABEL = {"wide": "Wide", "narrow": "Narrow", "sr2x": "Super-Resolve 2×", "altlines": "Altern. Lines", "half": "Half",
              "expand": "Expand", "box": "Box", "face_lower": "Lower face", "file": "Your mask"}
cycle = ["narrow", "expand", "half", "wide"]; k_other = 0
for i, it in enumerate(ITEMS):
    mfile = None
    if MASK_DIR:
        hits = [p for p in Path(MASK_DIR).rglob(it["name"] + ".*")]
        mfile = hits[0] if hits else None
    if mfile is not None:
        m = center_crop_arr(Image.open(mfile).convert("RGB"))[..., 0] / 255.0
        m = (m > 0.5).astype(np.float32); it["mask"] = m if MASK_WHITE_MEANS == "known" else 1 - m; it["mask_kind"] = "file"
    else:
        kind = PER_IMAGE_MASK.get(it["name"], DEFAULT_MASK)
        if kind == "auto":
            if it["domain"] == "face": kind = "face_lower"
            else: kind = cycle[k_other % len(cycle)]; k_other += 1
        it["mask"] = make_mask(kind, SEED + i); it["mask_kind"] = kind

X0 = torch.stack([torch.from_numpy(it["img"].astype(np.float32) / 127.5 - 1).permute(2, 0, 1) for it in ITEMS])
def to_np(t): return ((t.float().clamp(-1, 1) + 1) * 127.5).round().byte().permute(1, 2, 0).cpu().numpy()
TINT = np.array([125, 135, 255], np.float32)
def paper_input(img, m):
    """Masked input in the paper's style: missing pixels drawn as a translucent blue overlay."""
    m3 = m[..., None]; return (img * m3 + (1 - m3) * (0.35 * img + 0.65 * TINT)).astype(np.uint8)

fig, axes = plt.subplots(2, len(ITEMS), figsize=(2.6 * len(ITEMS), 5.4)); axes = np.atleast_2d(axes).reshape(2, -1)
for i, it in enumerate(ITEMS):
    axes[0, i].imshow(it["img"]); axes[0, i].set_title(f"{it['name']}  [{it['domain']}]", fontsize=9)
    axes[1, i].imshow(paper_input(it["img"], it["mask"])); axes[1, i].set_title(f"mask: {MASK_LABEL.get(it['mask_kind'], it['mask_kind'])}", fontsize=9)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); save_fig(fig, "inputs/images_and_masks", pdf=False)

## 4 · Priors: download on demand, load per GPU
Only the checkpoints needed by the selected images are fetched. Any checkpoint already present under `/kaggle/input` is used directly; attach one as a dataset if Google Drive's download quota blocks `gdown`.

In [ ]:
import gdown
from guided_diffusion.script_util import create_model, create_classifier

def find_input(name):
    for root in (INPUT_ROOT, ASSETS):
        if root.exists():
            hits = sorted(root.rglob(name))
            if hits: return hits[0]
    return None

def fetch_url(name):
    p = find_input(name)
    if p: return p
    dst = ASSETS / name
    try:
        print(f"Downloading {name} ..."); urllib.request.urlretrieve(OPENAI_URL + name, dst); return dst
    except Exception as e:
        print(f"  failed: {e}"); dst.unlink(missing_ok=True); return None

_drive_done = False
def fetch_drive(name):
    global _drive_done
    p = find_input(name)
    if p or _drive_done: return p
    for fid in GDRIVE_CKPT_IDS:                 # the two ids are celeba256 and places256; keep both
        try:
            print(f"gdown {fid} ..."); gdown.download(f"https://drive.google.com/uc?id={fid}", output=str(ASSETS) + "/", quiet=False)
        except Exception as e:
            print(f"  gdown failed: {str(e)[:150]}")
    _drive_done = True
    return find_input(name)

ADM = dict(image_size=256, num_channels=256, num_res_blocks=2, channel_mult="", learn_sigma=True, use_checkpoint=False,
           attention_resolutions="32,16,8", num_heads=4, num_head_channels=64, num_heads_upsample=-1,
           use_scale_shift_norm=True, dropout=0.0, resblock_updown=True, use_new_attention_order=False)

def resolve_domain(dom):
    """Return (domain_used, spec) with fallbacks."""
    if dom == "face":
        p = fetch_drive("celeba256_250000.pt")
        return ("face", dict(kind="adm", path=p, class_cond=False)) if p else ("face", dict(kind="hf", mid="google/ddpm-ema-celebahq-256"))
    if dom == "places":
        p = fetch_drive("places256_300000.pt")
        if p: return "places", dict(kind="adm", path=p, class_cond=False)
        print("places checkpoint unavailable -> imagenet"); dom = "imagenet"
    if dom == "imagenet":
        p, c = fetch_url("256x256_diffusion.pt"), fetch_url("256x256_classifier.pt")
        if p: return "imagenet", dict(kind="adm", path=p, class_cond=True, classifier=c)
        dom = "imagenet_uncond"
    p = fetch_url("256x256_diffusion_uncond.pt")
    if p: return "imagenet_uncond", dict(kind="adm", path=p, class_cond=False)
    raise RuntimeError("No usable checkpoint for " + dom)

SPECS = {}
for it in ITEMS:
    if it["domain"] not in SPECS: SPECS[it["domain"]] = resolve_domain(it["domain"])
    it["domain_used"], _ = SPECS[it["domain"]]
print({k: (v[0], v[1]["kind"], str(v[1].get("path", v[1].get("mid")))) for k, v in SPECS.items()})

class Prior:
    def __init__(self, spec, device):
        self.spec, self.device, self.kind = spec, device, spec["kind"]; self.classifier = None
        conf = types.SimpleNamespace(diffusion_steps=1000, use_value_logger=False)
        if self.kind == "adm":
            m = create_model(**ADM, class_cond=spec["class_cond"], use_fp16=USE_FP16, conf=conf)
            m.load_state_dict(torch.load(spec["path"], map_location="cpu"))
            if USE_FP16: m.convert_to_fp16()
            self.net = m.eval().requires_grad_(False).to(device)
            if spec.get("classifier"):
                c = create_classifier(image_size=256, classifier_use_fp16=False, classifier_width=128, classifier_depth=2,
                                      classifier_attention_resolutions="32,16,8", classifier_use_scale_shift_norm=True,
                                      classifier_resblock_updown=True, classifier_pool="attention")
                c.load_state_dict(torch.load(spec["classifier"], map_location="cpu"))
                # params must stay requires_grad=True: guided-diffusion's AttentionBlock always uses its own
                # CheckpointFunction, whose backward differentiates w.r.t. the parameters (as in official test.py)
                self.classifier = c.eval().to(device)
        else:
            from diffusers import UNet2DModel
            self.net = UNet2DModel.from_pretrained(spec["mid"]).eval().requires_grad_(False).to(device)
            if USE_FP16: self.net = self.net.half()
    @torch.no_grad()
    def __call__(self, x, t, y=None):
        if self.kind == "adm":
            out = self.net(x, t, y if self.spec["class_cond"] else None)
            return out[:, :3].float(), out[:, 3:].float()
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
            return self.net(x.half() if USE_FP16 else x, t).sample.float(), None
    def class_grad(self, x, t, y):
        """grad_x log p(y | x_t, t) * scale (official cond_fn)."""
        with torch.enable_grad():
            xi = x.detach().requires_grad_(True)
            lp = F.log_softmax(self.classifier(xi, t), dim=-1)[range(len(y)), y]
            return torch.autograd.grad(lp.sum(), xi)[0] * CLASSIFIER_SCALE

PRIORS = {}
def prior(domain, device):
    if (domain, device) not in PRIORS: PRIORS[(domain, device)] = Prior(SPECS[domain][1], device)
    return PRIORS[(domain, device)]

# ---- class labels for the class-conditional ImageNet model ----
try:
    from torchvision.models import ResNet50_Weights; IMAGENET_NAMES = ResNet50_Weights.IMAGENET1K_V1.meta["categories"]
except Exception:
    IMAGENET_NAMES = [str(i) for i in range(1000)]
for i, it in enumerate(ITEMS):
    it["y"] = None
    spec = SPECS[it["domain"]][1]
    if not spec.get("class_cond"): continue
    if isinstance(CLASS_LABEL, int): it["y"] = CLASS_LABEL
    elif CLASS_LABEL == "random": it["y"] = int(np.random.RandomState(SEED + i).randint(1000))
    else:   # auto: classify the visible context, with holes coarsely filled by OpenCV inpainting
        pr = prior(it["domain"], MAIN)
        if pr.classifier is None: it["y"] = int(np.random.RandomState(SEED + i).randint(1000))
        else:
            fill = cv2.inpaint(it["img"], ((1 - it["mask"]) * 255).astype(np.uint8), 5, cv2.INPAINT_TELEA)
            x = torch.from_numpy(fill.astype(np.float32) / 127.5 - 1).permute(2, 0, 1)[None].to(MAIN)
            with torch.no_grad(): it["y"] = int(pr.classifier(x, torch.zeros(1, dtype=torch.long, device=MAIN)).argmax())
    print(f"{it['name']}: class {it['y']} ({IMAGENET_NAMES[it['y']]})")

## 5 · RePaint sampler (paper Algorithm 1 + official jump schedule)
This is the same verified sampler as the benchmark notebook, plus two additions:
* **Classifier guidance** for the class-conditional ImageNet prior: $\mu \leftarrow \mu + \Sigma\,\nabla_{x_t}\log p_\phi(y\mid x_t)$, exactly as the official `condition_mean`.
* **Snapshot capture** at fractions of the schedule (Fig. 1's "Denoising 0 % / 60 % / 75 %") and GIF frames that include the forward jumps.

In [ ]:
BASE_AC = torch.cumprod(1 - torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64), 0)     # linear schedule (all ADM priors)

def space_timesteps(num_train, n):
    if n >= num_train: return list(range(num_train))
    frac, cur, out = (num_train - 1) / (n - 1), 0.0, []
    for _ in range(n): out.append(round(cur)); cur += frac
    return sorted(set(out))

def get_schedule_jump(t_T, n_sample=1, jump_length=10, jump_n_sample=10):
    jumps = {j: jump_n_sample - 1 for j in range(0, t_T - jump_length, jump_length)}
    t, ts = t_T, []
    while t >= 1:
        t -= 1; ts.append(t)
        if jumps.get(t, 0) > 0:
            jumps[t] -= 1
            for _ in range(jump_length): t += 1; ts.append(t)
    ts.append(-1); return ts
def n_calls(T, j, r): ts = get_schedule_jump(T, 1, j, r); return sum(b < a for a, b in zip(ts[:-1], ts[1:]))

_RS = {}
def respaced(T, dev):
    if (T, str(dev)) not in _RS:
        use = space_timesteps(1000, T); ac = BASE_AC[use]; acp = torch.cat([torch.ones(1, dtype=ac.dtype), ac[:-1]]); b = 1 - ac / acp
        pv = b * (1 - acp) / (1 - ac)
        d = dict(sa=ac.sqrt(), s1a=(1 - ac).sqrt(), sap=acp.sqrt(), s1ap=(1 - acp).sqrt(), c0=b * acp.sqrt() / (1 - ac),
                 ct=(1 - acp) * (1 - b).sqrt() / (1 - ac), plv=torch.log(torch.cat([pv[1:2], pv[1:]])), lb=torch.log(b),
                 sb=b.sqrt(), s1b=(1 - b).sqrt())
        d = {k: v.float().to(dev) for k, v in d.items()}; d["tm"] = torch.tensor(use, dtype=torch.long, device=dev); _RS[(T, str(dev))] = d
    return _RS[(T, str(dev))]

class Noise:
    def __init__(self, seeds, shape, dev): self.g = [torch.Generator(device=dev).manual_seed(int(s)) for s in seeds]; self.shape, self.dev = shape, dev
    def __call__(self): return torch.stack([torch.randn(self.shape, generator=g, device=self.dev) for g in self.g])

@torch.no_grad()
def repaint_sample(pr, x0, mask, seeds, y, T, j, r, snap_fracs=(), gif_frames=0, counter=None):
    dev, B = x0.device, x0.shape[0]; S = respaced(T, dev); noise = Noise(seeds, x0.shape[1:], dev)
    ts = get_schedule_jump(T, 1, j, r); pairs = list(zip(ts[:-1], ts[1:])); n = len(pairs)
    snap_at = {int(round(f * n)): f for f in snap_fracs}
    gif_at = set(np.linspace(0, n, gif_frames).round().astype(int).tolist()) if gif_frames else set()
    snaps, frames = {}, []
    x = noise()
    for i, (tl, tc) in enumerate(pairs):
        if i in snap_at: snaps[snap_at[i]] = x.half().cpu()
        if i in gif_at: frames.append((tl, x.half().cpu()))
        if tc < tl:                                                        # reverse step (Eq. 8)
            t = tl; tt = S["tm"][t].expand(B)
            eps, v = pr(x, tt, y)
            x0h = ((x - S["s1a"][t] * eps) / S["sa"][t]).clamp(-1, 1)
            mean = S["c0"][t] * x0h + S["ct"][t] * x
            if t > 0:
                lv = S["plv"][t] if v is None else ((v + 1) / 2) * S["lb"][t] + (1 - (v + 1) / 2) * S["plv"][t]
                if pr.classifier is not None and y is not None:
                    mean = mean + torch.exp(lv) * pr.class_grad(x, tt, y)
                xu = mean + torch.exp(0.5 * lv) * noise()
                xk = S["sap"][t] * x0 + S["s1ap"][t] * noise()
            else:
                xu, xk = mean, x0
            x = mask * xk + (1 - mask) * xu
            if counter: counter.update(B)
        else:                                                              # resampling jump (forward one step)
            x = S["s1b"][tc] * x + S["sb"][tc] * noise()
    if 1.0 in snap_fracs: snaps[1.0] = x.half().cpu()
    frames.append((-1, x.half().cpu()))
    return x.clamp(-1, 1), snaps, frames

print(f"Paper schedule: {n_calls(250, 10, 10)} network evaluations per sample")

## 6 · Runner: batching across GPUs, time budget, cache

In [ ]:
CACHE = OUT / "cache" / "store.pt"
STORE = torch.load(CACHE) if CACHE.exists() else {"x": {}, "snaps": {}, "frames": {}}
for f in (sorted(INPUT_ROOT.rglob("store.pt")) if INPUT_ROOT.exists() else []):
    if f != CACHE:
        o = torch.load(f); [STORE[k].update(o.get(k, {})) for k in STORE]; print("resumed", f)

# Masks per (image, mask kind); main masks are the per-image ones
MASKS = {(i, it["mask_kind"]): torch.from_numpy(it["mask"])[None] for i, it in enumerate(ITEMS)}
def get_mask(i, kind):
    if (i, kind) not in MASKS: MASKS[(i, kind)] = torch.from_numpy(make_mask(kind, SEED + 100 * i + MASK_TYPES.index(kind)))[None]
    return MASKS[(i, kind)]

def key(i, kind, s, T=None, j=None, r=None):
    return (ITEMS[i]["name"], kind, int(s), T or T_STEPS, j or JUMP_LENGTH, r if r is not None else JUMP_N_SAMPLE, ITEMS[i]["domain_used"])

def sec_per_image_step(domain):
    pr = prior(domain, MAIN); x = torch.randn(2, 3, 256, 256, device=MAIN); t = torch.full((2,), 500, device=MAIN)
    y = torch.zeros(2, dtype=torch.long, device=MAIN) if SPECS[domain][1].get("class_cond") else None
    pr(x, t, y); (pr.class_grad(x, t, y) if pr.classifier is not None else None)
    if IS_CUDA: torch.cuda.synchronize()
    t0 = time.time(); pr(x, t, y); (pr.class_grad(x, t, y) if pr.classifier is not None else None)
    if IS_CUDA: torch.cuda.synchronize()
    return (time.time() - t0) / 2
SPEED = {d: sec_per_image_step(d) for d in SPECS}
if MAX_BATCH_PER_GPU is None: MAX_BATCH_PER_GPU = 8 if IS_CUDA else 4
print("seconds per image-step:", {k: round(v, 3) for k, v in SPEED.items()}, "| batch/GPU:", MAX_BATCH_PER_GPU)

def run_jobs(jobs, label, snap_fracs=(), gif_keys=()):
    """jobs: list of (item_idx, mask_kind, seed, T, j, r)."""
    todo = [jb for jb in jobs if key(jb[0], jb[1], jb[2], *jb[3:]) not in STORE["x"]]
    if not todo: print(f"[{label}] cached"); return
    est = sum(n_calls(*jb[3:]) * SPEED[ITEMS[jb[0]]["domain"]] for jb in todo) / len(DEVICES)
    print(f"[{label}] {len(todo)} samples, ~{est/60:.0f} min (budget left {hours_left()*60:.0f} min)")
    if est / 3600 > hours_left(): print(f"[{label}] SKIPPED (time budget)"); return
    groups = {}
    for jb in todo: groups.setdefault((ITEMS[jb[0]]["domain"],) + tuple(jb[3:]), []).append(jb)
    units = [(g, js[s:s + MAX_BATCH_PER_GPU]) for g, js in groups.items() for s in range(0, len(js), MAX_BATCH_PER_GPU)]
    units.sort(key=lambda u: -len(u[1]) * n_calls(*u[0][1:]))
    q = queue.Queue(); [q.put(u) for u in units]; lock = threading.Lock(); done = [0]; errs = []
    class C:
        def update(self, n):
            with lock: done[0] += n
    def worker(dev):
        try:
            while True:
                try: (dom, T, j, r), js = q.get_nowait()
                except queue.Empty: return
                pr = prior(dom, dev)
                x0 = torch.stack([X0[jb[0]] for jb in js]).to(dev); m = torch.stack([get_mask(jb[0], jb[1]) for jb in js]).to(dev)
                ys = [ITEMS[jb[0]]["y"] for jb in js]
                y = torch.tensor(ys, device=dev) if ys[0] is not None else None
                want_gif = any(key(jb[0], jb[1], jb[2], *jb[3:]) in gif_keys for jb in js)
                out, snaps, frames = repaint_sample(pr, x0, m, [jb[2] for jb in js], y, T, j, r, snap_fracs,
                                                    gif_frames=72 if want_gif else 0, counter=C())
                with lock:
                    for b, jb in enumerate(js):
                        k = key(jb[0], jb[1], jb[2], *jb[3:]); STORE["x"][k] = out[b].half().cpu()
                        if snaps: STORE["snaps"][k] = {f: s[b] for f, s in snaps.items()}
                        if k in gif_keys: STORE["frames"][k] = [(t, fr[b]) for t, fr in frames]
        except Exception as e: errs.append(e)
    th = [threading.Thread(target=worker, args=(d,), daemon=True) for d in DEVICES]; [t.start() for t in th]
    total = sum(len(js) * n_calls(*g[1:]) for g, js in units); t0 = time.time()
    with tqdm(total=total, desc=label, unit="img-step", mininterval=3) as pb:
        while any(t.is_alive() for t in th): time.sleep(2); pb.n = min(done[0], total); pb.refresh()
    if errs: raise errs[0]
    torch.save(STORE, CACHE); wlog({f"perf/minutes_{label}": (time.time() - t0) / 60})
    print(f"[{label}] done in {(time.time()-t0)/60:.1f} min")

def seed_of(i, s): return SEED + 1000 * i + s
MAIN_JOBS = [(i, it["mask_kind"], seed_of(i, s), T_STEPS, JUMP_LENGTH, JUMP_N_SAMPLE) for i, it in enumerate(ITEMS) for s in range(N_SAMPLES)]
GIF_KEYS = {key(i, it["mask_kind"], seed_of(i, 0)) for i, it in enumerate(ITEMS)} if MAKE_GIFS else set()
R1_JOBS = [(i, it["mask_kind"], seed_of(i, 0), T_STEPS, JUMP_LENGTH, 1) for i, it in enumerate(ITEMS)]
GAL_JOBS = [(GALLERY_IMAGE, k, seed_of(GALLERY_IMAGE, 100 + s), T_STEPS, JUMP_LENGTH, JUMP_N_SAMPLE) for k in MASK_TYPES for s in range(GALLERY_SAMPLES)]
plan = pd.DataFrame([("A: Fig.1 samples", len(MAIN_JOBS), sum(n_calls(*jb[3:]) * SPEED[ITEMS[jb[0]]["domain"]] for jb in MAIN_JOBS) / len(DEVICES) / 60),
                     ("C: r=1 comparison", len(R1_JOBS) * RUN_RESAMPLING_COMPARISON, sum(n_calls(*jb[3:]) * SPEED[ITEMS[jb[0]]["domain"]] for jb in R1_JOBS) / len(DEVICES) / 60 * RUN_RESAMPLING_COMPARISON),
                     ("B: mask gallery", len(GAL_JOBS) * RUN_MASK_GALLERY, sum(n_calls(*jb[3:]) * SPEED[ITEMS[jb[0]]["domain"]] for jb in GAL_JOBS) / len(DEVICES) / 60 * RUN_MASK_GALLERY)],
                    columns=["experiment", "samples", "est. minutes"])
display(plan.round(0))

## 7 · Generate

In [ ]:
run_jobs(MAIN_JOBS, "fig1_samples", snap_fracs=tuple(DENOISE_SNAPSHOTS), gif_keys=GIF_KEYS)
if RUN_RESAMPLING_COMPARISON: run_jobs(R1_JOBS, "no_resampling_r1")
if RUN_MASK_GALLERY: run_jobs(GAL_JOBS, "mask_gallery")
for k, v in STORE["x"].items():
    Image.fromarray(to_np(v)).save(SAMPLES / f"{k[0]}_{k[1]}_seed{k[2]}_r{k[5]}.png")

## Figure A — paper Fig. 1 layout
*Input → Denoising 0 % → 60 % → 75 % → Sample 1 … Sample N.* The denoising columns show the composite $x_t$ of sample 1 at that fraction of the full (jumping) schedule. The samples are independent runs on the same input. Saved as PNG and **vector PDF**.

In [ ]:
def figure_A():
    ncol = 1 + len(DENOISE_SNAPSHOTS) + N_SAMPLES; nrow = len(ITEMS)
    fig, axes = plt.subplots(nrow, ncol, figsize=(1.45 * ncol, 1.45 * nrow + 0.35), gridspec_kw=dict(wspace=0.02, hspace=0.02))
    axes = np.atleast_2d(axes)
    for i, it in enumerate(ITEMS):
        k0 = key(i, it["mask_kind"], seed_of(i, 0))
        axes[i, 0].imshow(paper_input(it["img"], it["mask"]))
        for c, f in enumerate(DENOISE_SNAPSHOTS):
            s = STORE["snaps"].get(k0, {}).get(f)
            axes[i, 1 + c].imshow(to_np(s) if s is not None else np.full((256, 256, 3), 255, np.uint8))
        for s in range(N_SAMPLES):
            kk = key(i, it["mask_kind"], seed_of(i, s))
            axes[i, 1 + len(DENOISE_SNAPSHOTS) + s].imshow(to_np(STORE["x"][kk]) if kk in STORE["x"] else np.full((256, 256, 3), 255, np.uint8))
    labels = ["Input"] + [f"Denoising {int(round(f * 100))}%" for f in DENOISE_SNAPSHOTS] + [f"Sample {s + 1}" for s in range(N_SAMPLES)]
    for c, lab in enumerate(labels): axes[-1, c].set_xlabel(lab, fontsize=8)
    for a in axes.ravel(): a.set_xticks([]); a.set_yticks([]); [sp.set_visible(False) for sp in a.spines.values()]
    return fig
fig = figure_A(); save_fig(fig, "figures/A_fig1_style")

## Figure B — one image under every mask type (Figs. 5–7 style)

In [ ]:
if RUN_MASK_GALLERY:
    i = GALLERY_IMAGE; it = ITEMS[i]
    fig, axes = plt.subplots(len(MASK_TYPES), 2 + GALLERY_SAMPLES, figsize=(1.6 * (2 + GALLERY_SAMPLES), 1.6 * len(MASK_TYPES)),
                             gridspec_kw=dict(wspace=0.02, hspace=0.02))
    for r_, k in enumerate(MASK_TYPES):
        m = get_mask(i, k)[0].numpy()
        axes[r_, 0].imshow(paper_input(it["img"], m)); axes[r_, 0].set_ylabel(MASK_LABEL[k], fontsize=8)
        for s in range(GALLERY_SAMPLES):
            kk = key(i, k, seed_of(i, 100 + s))
            if kk in STORE["x"]: axes[r_, 1 + s].imshow(to_np(STORE["x"][kk]))
        axes[r_, -1].imshow(it["img"])
    for c, lab in enumerate(["Input"] + [f"RePaint {s + 1}" for s in range(GALLERY_SAMPLES)] + ["Original"]): axes[-1, c].set_xlabel(lab, fontsize=8)
    for a in axes.ravel(): a.set_xticks([]); a.set_yticks([]); [sp.set_visible(False) for sp in a.spines.values()]
    save_fig(fig, "figures/B_mask_gallery")

## Figure C — what resampling buys (Sec. 4.2 / Fig. 3)
The two runs share the initial noise, so the only difference is resampling. Without it, the generated region is locally plausible but often **inconsistent** with the context: seams, mismatched texture, wrong semantics.

In [ ]:
if RUN_RESAMPLING_COMPARISON:
    fig, axes = plt.subplots(len(ITEMS), 4, figsize=(6.4, 1.6 * len(ITEMS)), gridspec_kw=dict(wspace=0.02, hspace=0.02)); axes = np.atleast_2d(axes)
    for i, it in enumerate(ITEMS):
        k1, k10 = key(i, it["mask_kind"], seed_of(i, 0), r=1), key(i, it["mask_kind"], seed_of(i, 0))
        for c, im in enumerate([paper_input(it["img"], it["mask"]), STORE["x"].get(k1), STORE["x"].get(k10), it["img"]]):
            if im is not None: axes[i, c].imshow(im if isinstance(im, np.ndarray) else to_np(im))
    for c, lab in enumerate(["Input", "No resampling (r=1)", f"RePaint (r={JUMP_N_SAMPLE})", "Original"]): axes[-1, c].set_xlabel(lab, fontsize=8)
    for a in axes.ravel(): a.set_xticks([]); a.set_yticks([]); [sp.set_visible(False) for sp in a.spines.values()]
    save_fig(fig, "figures/C_resampling_effect")

## Figure D — the reverse process as an animation
Each frame is the composite $x_t$; the label shows the current diffusion time. Watch t jump back up: that is RePaint's resampling.

In [ ]:
from IPython.display import Image as IPImage, display
if MAKE_GIFS:
    for i, it in enumerate(ITEMS):
        kk = key(i, it["mask_kind"], seed_of(i, 0)); fr = STORE["frames"].get(kk)
        if not fr: continue
        S = respaced(T_STEPS, torch.device("cpu"))
        imgs = []
        for t, x in fr:
            a = np.ascontiguousarray(to_np(x)); lab = "final" if t < 0 else f"t={int(S['tm'][t])}"
            cv2.putText(a, lab, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(a, lab, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
            imgs.append(np.concatenate([paper_input(it["img"], it["mask"]), a], 1))
        imgs += [imgs[-1]] * 12
        p = OUT / "gifs" / f"{it['name']}_denoising.gif"; imageio.mimsave(p, imgs, duration=0.08, loop=0)
        wlog({f"gifs/{it['name']}": wandb.Video(str(p), format="gif")}); display(IPImage(filename=str(p)))

## Figure E — diversity and fidelity
* **Fidelity:** LPIPS between each sample and the original image (lower = closer to the original).
* **Diversity:** mean pairwise LPIPS between the N samples of the same input (higher = more varied completions).

Large holes (Expand, Half) should show high diversity. Small or structured holes (Super-Resolve, Alternating Lines) should show low diversity and high fidelity, as in the paper's discussion.

In [ ]:
import lpips
LP = lpips.LPIPS(net="alex", verbose=False).to(MAIN).eval()
rows = []
with torch.no_grad():
    for i, it in enumerate(ITEMS):
        xs = [STORE["x"][key(i, it["mask_kind"], seed_of(i, s))].float() for s in range(N_SAMPLES) if key(i, it["mask_kind"], seed_of(i, s)) in STORE["x"]]
        if not xs: continue
        X = torch.stack(xs).to(MAIN); G = X0[i:i + 1].to(MAIN).expand_as(X)
        fid = LP(X, G).flatten().cpu().numpy()
        pair = [float(LP(X[a:a + 1], X[b:b + 1])) for a in range(len(xs)) for b in range(a + 1, len(xs))]
        rows.append(dict(image=it["name"], domain=it["domain_used"], mask=MASK_LABEL.get(it["mask_kind"]), missing_pct=100 * (1 - it["mask"].mean()),
                         lpips_to_original=float(fid.mean()), diversity_pairwise_lpips=float(np.mean(pair)) if pair else np.nan))
DIV = pd.DataFrame(rows); display(DIV.round(3)); DIV.to_csv(OUT / "diversity_fidelity.csv", index=False)
wlog({"metrics/diversity_fidelity": wandb.Table(dataframe=DIV)})
if len(DIV):
    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    ax.scatter(DIV.lpips_to_original, DIV.diversity_pairwise_lpips, s=60, color="#2a78d6", edgecolor="white", zorder=3)
    for _, r_ in DIV.iterrows(): ax.annotate(f"{r_.image} ({r_['mask']})", (r_.lpips_to_original, r_.diversity_pairwise_lpips), xytext=(5, 4), textcoords="offset points", fontsize=8)
    ax.set(xlabel="LPIPS to original  (fidelity, lower = closer)", ylabel="pairwise LPIPS between samples  (diversity)", title="Diversity vs. fidelity per input")
    ax.grid(alpha=0.25); plt.tight_layout(); save_fig(fig, "figures/E_diversity_fidelity", pdf=False)

## Download

In [ ]:
(OUT / "run_config.json").write_text(json.dumps(dict(
    images=[{k: v for k, v in it.items() if k not in ("img", "mask")} for it in ITEMS],
    specs={k: [v[0], {kk: str(vv) for kk, vv in v[1].items()}] for k, v in SPECS.items()},
    T=T_STEPS, j=JUMP_LENGTH, r=JUMP_N_SAMPLE, n_samples=N_SAMPLES, class_label=str(CLASS_LABEL), seed=SEED), indent=2, default=str))
if wandb.run is not None:
    art = wandb.Artifact("repaint-figures", type="results"); art.add_dir(str(FIG), name="figures"); art.add_dir(str(SAMPLES), name="samples")
    if any((OUT / "gifs").iterdir()): art.add_dir(str(OUT / "gifs"), name="gifs")
    run.log_artifact(art)
WRUN = Path(run.dir).parent; wandb.finish()
INCLUDE_CACHE = False     # True adds the raw tensor cache (large; only needed to resume in a later session)
zp = WORKING_ROOT / "repaint_figures.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.rglob("*")):
        if f.is_file() and (INCLUDE_CACHE or (OUT / "cache") not in f.parents): zf.write(f, f.relative_to(OUT))
    if WANDB_MODE == "offline":
        for f in WRUN.rglob("*"):
            if f.is_file(): zf.write(f, Path("wandb_offline_run") / f.relative_to(WRUN))
print(f"{zp}: {zp.stat().st_size/2**20:.1f} MB")
from IPython.display import FileLink
os.chdir(WORKING_ROOT); display(FileLink(zp.name, result_html_prefix="Download: "))